In [0]:
import requests
import json
import boto3

In [0]:
%run ../helper_function/helper_function

In [0]:
access_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-access-key")
secret_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-secret-key")
databricks_token = dbutils.secrets.get(scope="barcakings-secrets",key="dbk-pat")

In [0]:
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
databricks_host = f"https://{workspace_url}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

In [0]:
dbutils.widgets.text("sport", "football", "sport")
sport = dbutils.widgets.get("sport")

In [0]:
# Read control table
control_df = spark.sql(
    f"""
        SELECT 
            *
        FROM  
            workspace.bk_raw.raw_control_tbl 
        WHERE 
            sport = '{sport}' 
    """)
control_df.display()

In [0]:
control_rows = control_df.collect()

for row in control_rows:
    team = row["team"]
    file_name = row["file_name"]
    api_url = row["api_url"]
    api_param = json.loads(row["api_param"])
    pn_flag = row["pn_flag"]
    offset = row["offset"]
    limit = row["limit"]
    volume_path = row["target_volume_path"]+file_name+".json"
    bucket_name = row["target_s3_bucket"]
    s3_key = row["target_s3_key"]+file_name+".json"

    if pn_flag:
        json_data = pn_api_extraction(api_url,api_param,headers,offset,limit)
    else:
        json_data = api_extraction(api_url,api_param)
    
    upload_s3_vol(json_data,bucket_name,s3_key,volume_path)